# **Feature Engineering**
**Decodelabs Internship | Week 2**

---
Here, I transformed the cleaned interim dataset into a rich, model-ready feature set.
This involves converting ICD-9 codes to disease categories, encoding age as an
ordinal variable, summarising 24 medication columns into aggregate features,
and creating new domain-informed features.


In [1]:
import sys, os
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from configs.config import (
    RAW_FILE, IDS_MAP_FILE, INTERIM_FILE, PROCESSED_FILE,
    TRAIN_FILE, VAL_FILE, TEST_FILE,
    FIGURES_DIR, TABLES_DIR, PAPER_FIG_DIR, PAPER_TAB_DIR,
    RANDOM_SEED, TARGET_COL, PATIENT_ID_COL, MEDICATION_COLS,
    AGE_ORDER, icd9_to_category, COLORS, ensure_dirs
)
from src.plot_utils import set_plot_style, save_figure, save_table
ensure_dirs()
set_plot_style()
print("Config loaded. Seed:", RANDOM_SEED)

Config loaded. Seed: 42


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv(INTERIM_FILE)
print(f"Loaded interim data: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Loaded interim data: 69,987 rows × 46 columns


,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary
0,135,Caucasian,Female,[50-60),2,1,1,8,77,6,...,No,Steady,No,No,No,No,No,Ch,Yes,1
1,378,Caucasian,Female,[50-60),3,1,1,2,49,1,...,No,No,No,No,No,No,No,No,No,0
2,729,Caucasian,Female,[80-90),1,3,7,4,68,2,...,No,No,No,No,No,No,No,No,Yes,0


## Converting ICD-9 diagnosis codes to disease categories

The three diagnosis columns (diag_1, diag_2, diag_3) contain raw ICD-9 codes. I mapped these to 9 broad disease categories using the `icd9_to_category()` function from `configs/config.py`. This reduces thousands of unique codes to 9 meaningful groups.

In [3]:
for diag_col in ["diag_1", "diag_2", "diag_3"]:
    new_col = diag_col + "_cat"
    df[new_col] = df[diag_col].apply(icd9_to_category)
    print(f"=== {new_col} value counts ===")
    print(df[new_col].value_counts().to_string())
    print()

# I dropped the original raw ICD-9 columns, now that we have categories
df.drop(columns=["diag_1", "diag_2", "diag_3"], inplace=True)
print("Dropped raw diag_1, diag_2, diag_3 columns.")
print(f"Shape: {df.shape[0]:,} × {df.shape[1]}")

=== diag_1_cat value counts ===
diag_1_cat
Circulatory         21119
Other               16752
Respiratory          9468
Digestive            6534
Injury               4817
Musculoskeletal      3950
Genitourinary        3548
Neoplasm             2514
Other / External     1085
Diabetes              200

=== diag_2_cat value counts ===
diag_2_cat
Circulatory         22008
Other               20849
Respiratory          7015
Genitourinary        5473
Diabetes             4897
Digestive            2897
Other / External     2113
Injury               1823
Neoplasm             1624
Musculoskeletal      1288

=== diag_3_cat value counts ===
diag_3_cat
Circulatory         20899
Other               20082
Diabetes             8855
Other / External     4704
Respiratory          4678
Genitourinary        4130
Digestive            2643
Injury               1423
Musculoskeletal      1381
Neoplasm             1192

Dropped raw diag_1, diag_2, diag_3 columns.
Shape: 69,987 × 46


## Encoded age as an ordinal integer

Age is stored as decade buckets: `[50-60)`, `[60-70)`, etc. I mapped these to integers 0–9 using the `AGE_ORDER` dict from `config`. This preserves the natural ordering: older age = higher number.

In [4]:
df["age_ordinal"] = df["age"].map(AGE_ORDER)

print("=== Age encoding check ===")
mapping_check = df[["age","age_ordinal"]].drop_duplicates().sort_values("age_ordinal")
print(mapping_check.to_string(index=False))

# Check no NaN values were introduced
n_nan = df["age_ordinal"].isnull().sum()
print(f"\nNaN values in age_ordinal: {n_nan}")

# Drop the original string age column
df.drop(columns=["age"], inplace=True)
print("Dropped original 'age' string column.")

=== Age encoding check ===
     age  age_ordinal
  [0-10)            0
 [10-20)            1
 [20-30)            2
 [30-40)            3
 [40-50)            4
 [50-60)            5
 [60-70)            6
 [70-80)            7
 [80-90)            8
[90-100)            9

NaN values in age_ordinal: 0
Dropped original 'age' string column.


## Encoded gender as binary

In [5]:
df["gender_binary"] = (df["gender"] == "Male").astype(int)
print("=== Gender encoding ===")
print(df[["gender","gender_binary"]].drop_duplicates())

df.drop(columns=["gender"], inplace=True)
print("\nDropped original 'gender' column.")

=== Gender encoding ===
   gender  gender_binary
0  Female              0
8    Male              1

Dropped original 'gender' column.


## Encoded medication columns as numeric

In [6]:
# The 24 medication columns have values: 'No', 'Steady', 'Up', 'Down'
# Meaning:
#   No     = this drug was not prescribed
#   Steady = dosage unchanged
#   Up     = dosage increased
#   Down   = dosage decreased
#
# I encoded these as: No=0, Steady=1, Up=2, Down=3 (ordinal by change magnitude)

MED_MAP = {"No": 0, "Steady": 1, "Down": 2, "Up": 3}

for col in MEDICATION_COLS:
    if col in df.columns:
        df[col] = df[col].map(MED_MAP)

# Verify encoding
print("=== Medication encoding check (insulin) ===")
print(df["insulin"].value_counts().sort_index())
print()

# Check for NaN introduced by unmapped values
med_nan = df[MEDICATION_COLS].isnull().sum().sum()
print(f"NaN values introduced in medication columns: {med_nan}")

=== Medication encoding check (insulin) ===
insulin
0    33925
1    21690
2     7490
3     6882
Name: count, dtype: int64

NaN values introduced in medication columns: 0


## Creating aggregate medication features

Instead of 24 individual drug columns, I created summary features that capture the overall medication picture in fewer, more informative dimensions.

In [7]:
# Total number of medications being used (> 0 = prescribed)
df["n_medications_active"] = (df[MEDICATION_COLS] > 0).sum(axis=1)

# Number of medication changes (Up or Down = values 2 or 3)
df["n_medication_changes"] = (df[MEDICATION_COLS] >= 2).sum(axis=1)

# Number of medication increases specifically
df["n_medication_increases"] = (df[MEDICATION_COLS] == 3).sum(axis=1)

print("=== New medication aggregate features ===")
for col in ["n_medications_active", "n_medication_changes", "n_medication_increases"]:
    print(f"  {col:30s}  mean={df[col].mean():.2f}  max={df[col].max()}")

# Example: are more medication changes associated with higher readmission?
for col in ["n_medications_active", "n_medication_changes"]:
    mean_no  = df[df[TARGET_COL]==0][col].mean()
    mean_yes = df[df[TARGET_COL]==1][col].mean()
    print(f"\n  {col}")
    print(f"    Mean (no readmit): {mean_no:.2f}")
    print(f"    Mean (readmit)   : {mean_yes:.2f}")

=== New medication aggregate features ===
  n_medications_active            mean=1.19  max=6
  n_medication_changes            mean=0.26  max=4
  n_medication_increases          mean=0.14  max=3

  n_medications_active
    Mean (no readmit): 1.18
    Mean (readmit)   : 1.21

  n_medication_changes
    Mean (no readmit): 0.26
    Mean (readmit)   : 0.31


## Creating comorbidity count feature

`number_diagnoses` already exists in the dataset, but I also created a feature for whether the primary diagnosis is diabetes itself.

In [8]:
df["primary_diag_is_diabetes"] = (df["diag_1_cat"] == "Diabetes").astype(int)

print("=== primary_diag_is_diabetes distribution ===")
print(df["primary_diag_is_diabetes"].value_counts())

# Cross-tab with target
ct = pd.crosstab(df["primary_diag_is_diabetes"], df[TARGET_COL],
                 normalize="index") * 100
ct.columns = ["No Readmit %", "Readmit %"]
ct.index = ["Non-diabetes primary dx", "Diabetes primary dx"]
print()
print(ct.round(1))

=== primary_diag_is_diabetes distribution ===
primary_diag_is_diabetes
0    69787
1      200
Name: count, dtype: int64

                         No Readmit %  Readmit %
Non-diabetes primary dx          92.6        7.4
Diabetes primary dx              95.5        4.5


## Creating hospital utilisation features

Prior utilisation (outpatient, emergency, inpatient visits in the past year) is a strong clinical indicator of readmission risk. I created a total prior utilisation feature.

In [9]:
df["total_prior_visits"] = (df["number_outpatient"] +
                             df["number_emergency"] +
                             df["number_inpatient"])

print("=== total_prior_visits distribution ===")
print(df["total_prior_visits"].describe().round(2))

# Indicator: any prior inpatient admission
df["had_prior_inpatient"] = (df["number_inpatient"] > 0).astype(int)
print(f"\nPatients with any prior inpatient admission: {df['had_prior_inpatient'].sum():,} "
      f"({df['had_prior_inpatient'].mean()*100:.1f}%)")

=== total_prior_visits distribution ===
count    69987.00
mean         0.76
std          1.67
min          0.00
25%          0.00
50%          0.00
75%          1.00
max         61.00
Name: total_prior_visits, dtype: float64

Patients with any prior inpatient admission: 14,764 (21.1%)


## Encode A1C result and glucose serum

In [10]:
# A1Cresult: '>8', '>7', 'Norm', 'None' (not measured)
# max_glu_serum: '>300', '>200', 'Norm', 'None'
# I encode these as ordinal + a flag for whether the test was done.

A1C_MAP      = {"Norm": 1, ">7": 2, ">8": 3}
GLU_MAP      = {"Norm": 1, ">200": 2, ">300": 3}

df["A1Cresult_encoded"]     = df["A1Cresult"].map(A1C_MAP).fillna(0).astype(int)
df["max_glu_serum_encoded"] = df["max_glu_serum"].map(GLU_MAP).fillna(0).astype(int)
df["A1C_tested"]     = df["A1Cresult"].notna().astype(int)
df["glucose_tested"] = df["max_glu_serum"].notna().astype(int)

df.drop(columns=["A1Cresult", "max_glu_serum"], inplace=True)

print("=== A1C result encoded ===")
print(df["A1Cresult_encoded"].value_counts().sort_index())
print()
print(f"A1C tested: {df['A1C_tested'].mean()*100:.1f}% of patients")
print(f"Glucose tested: {df['glucose_tested'].mean()*100:.1f}% of patients")

=== A1C result encoded ===
A1Cresult_encoded
0    57517
1     3724
2     2767
3     5979
Name: count, dtype: int64

A1C tested: 17.8% of patients
Glucose tested: 4.8% of patients


## Encoding change and diabetesMed

In [11]:
# change: 'Ch' = medication was changed, 'No' = no change
# diabetesMed: 'Yes' or 'No' — whether any diabetes medication was prescribed

df["med_changed"] = (df["change"] == "Ch").astype(int)
df["on_diabetes_med"] = (df["diabetesMed"] == "Yes").astype(int)

df.drop(columns=["change", "diabetesMed"], inplace=True)

print("=== med_changed ===")
print(df["med_changed"].value_counts())
print()
print("=== on_diabetes_med ===")
print(df["on_diabetes_med"].value_counts())

=== med_changed ===
med_changed
0    38359
1    31628
Name: count, dtype: int64

=== on_diabetes_med ===
on_diabetes_med
1    53311
0    16676
Name: count, dtype: int64


## Summary of engineered features

In [12]:
print("=== Feature Engineering Summary ===")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("=== All columns ===")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:3d}. {col}")

=== Feature Engineering Summary ===
Shape: 69,987 rows × 54 columns

=== All columns ===
    1. patient_nbr
    2. race
    3. admission_type_id
    4. discharge_disposition_id
    5. admission_source_id
    6. time_in_hospital
    7. num_lab_procedures
    8. num_procedures
    9. num_medications
   10. number_outpatient
   11. number_emergency
   12. number_inpatient
   13. number_diagnoses
   14. metformin
   15. repaglinide
   16. nateglinide
   17. chlorpropamide
   18. glimepiride
   19. acetohexamide
   20. glipizide
   21. glyburide
   22. tolbutamide
   23. pioglitazone
   24. rosiglitazone
   25. acarbose
   26. miglitol
   27. troglitazone
   28. tolazamide
   29. examide
   30. citoglipton
   31. insulin
   32. glyburide-metformin
   33. glipizide-metformin
   34. glimepiride-pioglitazone
   35. metformin-rosiglitazone
   36. metformin-pioglitazone
   37. readmitted_binary
   38. diag_1_cat
   39. diag_2_cat
   40. diag_3_cat
   41. age_ordinal
   42. gender_binary
   43. n

In [13]:
# Save the fully feature-engineered dataset
os.makedirs(os.path.dirname(PROCESSED_FILE), exist_ok=True)
df.to_csv(PROCESSED_FILE, index=False)
print(f"Processed dataset saved: {PROCESSED_FILE}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Processed dataset saved: c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\data\processed\diabetic_processed.csv
Shape: 69,987 rows × 54 columns
